In [ ]:
# PART 1 : IMPORT LIBRARIES

import os
import random
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version :", torch.__version__)
print("Device          :", DEVICE)

PyTorch Version : 2.11.0+cu128
Device          : cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive"

In [ ]:
# LOAD DATASET
# ----------------------------------------------------------------------
# Your Drive folder has several file groups (NEW_ROI_X*/NEW_ROI_Y*, X*/Y*,
# shape_*/roi_shape_*), plus Mask R-CNN checkpoints and image folders from
# what looks like an earlier / separate detection experiment.
#
# This notebook uses the shape_*/roi_shape_*/shape_labels.npy set:
#   shape_train.npy / shape_val.npy / shape_test.npy             -> images
#   roi_shape_train.npy / roi_shape_val.npy / roi_shape_test.npy -> masks
#   shape_labels.npy                                              -> shape labels
# because it's the only pre-split set that has a matching labels file
# (shape_labels.npy) -- NEW_ROI_* and X*/Y* have no label file at all,
# so they can't be used for shape classification. If that's wrong,
# just swap the filenames below.
# ----------------------------------------------------------------------

X_train = np.load(os.path.join(DATASET_PATH, "shape_train.npy"))
X_val   = np.load(os.path.join(DATASET_PATH, "shape_val.npy"))
X_test  = np.load(os.path.join(DATASET_PATH, "shape_test.npy"))

Y_train = np.load(os.path.join(DATASET_PATH, "roi_shape_train.npy"))
Y_val   = np.load(os.path.join(DATASET_PATH, "roi_shape_val.npy"))
Y_test  = np.load(os.path.join(DATASET_PATH, "roi_shape_test.npy"))

CLASS_NAMES = {0: "Irregular", 1: "Oval", 2: "Lobulated"}
NUM_CLASSES = len(CLASS_NAMES)

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_total = n_train + n_val + n_test

for name, imgs, masks in [("Train", X_train, Y_train), ("Val", X_val, Y_val), ("Test", X_test, Y_test)]:
    assert len(imgs) == len(masks), f"{name}: image/mask count mismatch ({len(imgs)} vs {len(masks)})"

# ----------------------------------------------------------------------
# SHAPE LABELS  (0: Irregular, 1: Oval, 2: Lobulated)
# ----------------------------------------------------------------------
# It wasn't confirmed whether shape_labels.npy covers ALL samples
# (train+val+test concatenated, in that order) or only the training
# split. This auto-detects which case applies by comparing lengths.
# If neither matches, it stops with a clear message instead of silently
# mis-aligning labels to images -- fix the logic below once you know
# how your labels are actually organized.

all_labels = np.load(os.path.join(DATASET_PATH, "shape_labels.npy")).astype(np.int64)

if len(all_labels) == n_total:
    print(f"shape_labels.npy has {len(all_labels)} entries -> matches TRAIN+VAL+TEST combined. Slicing per split.")
    L_train = all_labels[:n_train]
    L_val   = all_labels[n_train:n_train + n_val]
    L_test  = all_labels[n_train + n_val:]

elif len(all_labels) == n_train:
    raise ValueError(
        f"shape_labels.npy has {len(all_labels)} entries, matching only the "
        f"TRAINING split ({n_train} samples) -- there are no labels here for "
        f"val ({n_val}) or test ({n_test}). Classification needs labels for "
        f"every split. If separate label files exist for val/test "
        f"(e.g. 'shape_labels_val.npy', 'shape_labels_test.npy'), load and "
        f"assign them to L_val / L_test here instead."
    )

else:
    raise ValueError(
        f"shape_labels.npy has {len(all_labels)} entries, which matches "
        f"neither the combined total ({n_total}) nor the training split "
        f"alone ({n_train}). Inspect how your labels line up with "
        f"shape_train/val/test.npy and fix the slicing in this cell."
    )

for name, split_labels in [("Train", L_train), ("Val", L_val), ("Test", L_test)]:
    assert len(split_labels) == {"Train": n_train, "Val": n_val, "Test": n_test}[name], \
        f"{name}: label count does not match image count!"
    assert set(np.unique(split_labels)).issubset(set(CLASS_NAMES.keys())), \
        f"{name} labels must only contain {list(CLASS_NAMES.keys())}"

print("\nDataset loaded successfully!\n")
print(f"Train : Images {X_train.shape}, Masks {Y_train.shape}, Labels {L_train.shape}")
print(f"Val   : Images {X_val.shape}, Masks {Y_val.shape}, Labels {L_val.shape}")
print(f"Test  : Images {X_test.shape}, Masks {Y_test.shape}, Labels {L_test.shape}")

for name, split_labels in [("Train", L_train), ("Val", L_val), ("Test", L_test)]:
    counts = dict(zip(*np.unique(split_labels, return_counts=True)))
    readable = {CLASS_NAMES[k]: v for k, v in counts.items()}
    print(f"{name} label distribution : {readable}")


In [ ]:
print("Train Image dtype :", X_train.dtype, " Mask dtype :", Y_train.dtype, " Label dtype :", L_train.dtype)
print("Train Image Min/Max :", X_train.min(), X_train.max())
print("Mask Unique (Train)  :", np.unique(Y_train))
print("Label Unique (Train) :", np.unique(L_train), "->", [CLASS_NAMES[l] for l in np.unique(L_train)])


In [ ]:
import matplotlib.pyplot as plt
import random

plt.figure(figsize=(10, 10))

for i in range(5):
    idx = random.randint(0, len(X_train)-1)

    plt.subplot(5, 2, 2*i+1)
    plt.imshow(X_train[idx].squeeze(), cmap="gray")
    plt.title(f"Image (Shape: {CLASS_NAMES[int(L_train[idx])]})")
    plt.axis("off")

    plt.subplot(5, 2, 2*i+2)
    plt.imshow(Y_train[idx].squeeze(), cmap="gray")
    plt.title("Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# PART 2 : IMPORT LIBRARIES

from torch.utils.data import Dataset, DataLoader


In [ ]:
# TRAIN / VALIDATION / TEST SPLIT
# Data is already pre-split in your Drive folder (shape_train/val/test.npy
# + roi_shape_train/val/test.npy + shape_labels.npy), so there's nothing
# to split here -- this just confirms the sizes line up.

print(f"Training Images   : {len(X_train)}")
print(f"Validation Images : {len(X_val)}")
print(f"Testing Images    : {len(X_test)}")


In [ ]:
# CUSTOM PYTORCH DATASET
class BreastCancerDataset(Dataset):

    def __init__(self, images, masks, labels):
        self.images = images
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image = self.images[index]
        mask = self.masks[index]
        label = self.labels[index]

        # Convert NumPy -> Torch Tensor
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()
        label = torch.tensor(label).long()

        # Convert HWC -> CHW
        image = image.permute(2, 0, 1)
        mask = mask.permute(2, 0, 1)

        return image, mask, label


In [ ]:
# CREATE DATASETS

train_dataset = BreastCancerDataset(X_train, Y_train, L_train)
val_dataset = BreastCancerDataset(X_val, Y_val, L_val)
test_dataset = BreastCancerDataset(X_test, Y_test, L_test)


In [ ]:
# CREATE DATALOADERS

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# VERIFY DATALOADER

images, masks, labels_batch = next(iter(train_loader))

print(f"Images Shape : {images.shape}")
print(f"Masks Shape  : {masks.shape}")
print(f"Labels Shape : {labels_batch.shape}")

print(f"Image dtype  : {images.dtype}")
print(f"Mask dtype   : {masks.dtype}")
print(f"Label dtype  : {labels_batch.dtype}")

print(f"Device       : {DEVICE}")


In [ ]:
# RESIDUAL BLOCK

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv_block = nn.Sequential(

            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)

        )

        self.shortcut = nn.Sequential()

        if in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        x = self.conv_block(x)
        x = x + residual
        x = self.relu(x)
        return x

In [ ]:
# ENCODER BLOCK

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.residual = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        features = self.residual(x)
        pooled = self.pool(features)
        return features, pooled

In [ ]:
# DECODER BLOCK

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.residual = ResidualBlock(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.residual(x)
        return x

In [ ]:
# ORIGINAL RESUNET (pure segmentation)

class ResUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.e1 = EncoderBlock(1, 32)
        self.e2 = EncoderBlock(32, 64)
        self.e3 = EncoderBlock(64, 128)
        self.e4 = EncoderBlock(128, 256)

        # Bridge
        self.bridge = ResidualBlock(256, 512)

        # Decoder
        self.d4 = DecoderBlock(512, 256, 256)
        self.d3 = DecoderBlock(256, 128, 128)
        self.d2 = DecoderBlock(128, 64, 64)
        self.d1 = DecoderBlock(64, 32, 32)

        # Output
        self.output = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

    def forward(self, x):

        s1, p1 = self.e1(x)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        b = self.bridge(p4)

        d4 = self.d4(b, s4)
        d3 = self.d3(d4, s3)
        d2 = self.d2(d3, s2)
        d1 = self.d1(d2, s1)

        out = self.output(d1)

        return out


In [ ]:
# INITIALIZE MODEL

model = ResUNet().to(DEVICE)

print(model)


In [ ]:
# VERIFY MODEL
images, masks, _ = next(iter(train_loader))
images = images.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

print("Input Shape :", images.shape)
print("Output Shape:", outputs.shape)


In [ ]:
# MODEL PARAMETERS

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 8,113,601
Trainable Parameters : 8,113,601


In [ ]:
# PART 4 : LOSS FUNCTIONS & METRICS

import torch.nn.functional as F

In [ ]:
# DICE COEFFICIENT

def dice_coefficient(pred, target, smooth=1e-6):
    """
    Computes Dice Score.

    Parameters:
        pred   : Model predictions
        target : Ground truth mask

    Returns:
        Dice coefficient
    """

    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    intersection = (pred * target).sum()

    dice = (2.0 * intersection + smooth) / (
        pred.sum() + target.sum() + smooth
    )

    return dice

In [ ]:
# INTERSECTION OVER UNION

def iou_score(pred, target, smooth=1e-6):
    """
    Computes IoU Score.
    """

    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    intersection = (pred * target).sum()

    union = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou

In [ ]:
# DICE LOSS

class DiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):

        dice = dice_coefficient(pred, target)

        return 1 - dice

In [ ]:
class BCEDiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):

        bce_loss = self.bce(pred, target)

        # Apply sigmoid only for Dice computation
        pred = torch.sigmoid(pred)

        dice_loss = self.dice(pred, target)

        return bce_loss + dice_loss

In [ ]:
# LOSS FUNCTION

criterion = BCEDiceLoss()

print("Loss Function Initialized Successfully!")


In [ ]:
# METRICS HELPER FUNCTION
def calculate_metrics(outputs, masks):
    predictions = (torch.sigmoid(outputs) > 0.5).float()

    dice = dice_coefficient(predictions, masks)
    iou = iou_score(predictions, masks)

    return dice.item(), iou.item()


In [ ]:
# VERIFY LOSS FUNCTION

images, masks, _ = next(iter(train_loader))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

loss = criterion(outputs, masks)

dice, iou = calculate_metrics(outputs, masks)

print(f"Loss : {loss:.4f}")
print(f"Dice : {dice:.4f}")
print(f"IoU  : {iou:.4f}")


In [ ]:
# PART 5 : TRAINING SETUP

import copy
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler

In [ ]:
# TRAINING HYPERPARAMETERS

LEARNING_RATE = 1e-3
NUM_EPOCHS = 50

PATIENCE = 10

MODEL_PATH = "best_resunet.pth"

In [ ]:
# OPTIMIZER

optimizer = Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
# LEARNING RATE SCHEDULER

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
# MIXED PRECISION

scaler = GradScaler(enabled=torch.cuda.is_available())

print("Mixed Precision:", torch.cuda.is_available())

Mixed Precision: True


In [ ]:
# EARLY STOPPING

class EarlyStopping:

    def __init__(self, patience=10):

        self.patience = patience
        self.counter = 0

        self.best_loss = float("inf")

        self.early_stop = False

    def __call__(self, val_loss):

        if val_loss < self.best_loss:

            self.best_loss = val_loss
            self.counter = 0

        else:

            self.counter += 1

            print(f"EarlyStopping Counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:

                self.early_stop = True

In [ ]:
# INITIALIZE EARLY STOPPING

early_stopping = EarlyStopping(
    patience=PATIENCE
)

In [ ]:
# TRAINING HISTORY

history = {

    "train_loss": [],
    "val_loss": [],

    "train_dice": [],
    "val_dice": [],

    "train_iou": [],
    "val_iou": []

}


In [ ]:
# SAVE BEST MODEL

best_val_loss = float("inf")

def save_best_model(model, val_loss):

    global best_val_loss

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(model.state_dict(), MODEL_PATH)

        print(f"Best model saved! Validation Loss: {val_loss:.4f}")

In [ ]:
# VERIFY TRAINING SETUP

print("Optimizer      :", optimizer.__class__.__name__)
print("Scheduler      :", scheduler.__class__.__name__)
print("Loss Function  :", criterion.__class__.__name__)
print("Device         :", DEVICE)


In [ ]:
# PART 6 : TRAINING LOOP

from tqdm.auto import tqdm

In [ ]:
# TRAIN ONE EPOCH

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):

    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, masks, _ in progress_bar:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        if device.type == "cuda":

            with torch.cuda.amp.autocast():

                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        dice, iou = calculate_metrics(outputs, masks)

        running_loss += loss.item()
        running_dice += dice
        running_iou += iou

        progress_bar.set_postfix(
            Loss=f"{loss.item():.4f}",
            Dice=f"{dice:.4f}",
            IoU=f"{iou:.4f}"
        )

    return (
        running_loss / len(loader),
        running_dice / len(loader),
        running_iou / len(loader)
    )


In [ ]:
# VALIDATE ONE EPOCH

def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    with torch.no_grad():

        progress_bar = tqdm(loader, desc="Validation", leave=False)

        for images, masks, _ in progress_bar:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            dice, iou = calculate_metrics(outputs, masks)

            running_loss += loss.item()
            running_dice += dice
            running_iou += iou

            progress_bar.set_postfix(
                Loss=f"{loss.item():.4f}",
                Dice=f"{dice:.4f}",
                IoU=f"{iou:.4f}"
            )

    return (
        running_loss / len(loader),
        running_dice / len(loader),
        running_iou / len(loader)
    )


In [ ]:
# TRAIN MODEL

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print("-" * 60)

    train_loss, train_dice, train_iou = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE
    )

    val_loss, val_dice, val_iou = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)

    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")

    print(f"Train Dice : {train_dice:.4f}")
    print(f"Val Dice   : {val_dice:.4f}")

    print(f"Train IoU  : {train_iou:.4f}")
    print(f"Val IoU    : {val_iou:.4f}")

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(model.state_dict(), MODEL_PATH)

        print("Best model saved!")

    early_stopping(val_loss)

    if early_stopping.early_stop:

        print("\nEarly stopping triggered!")

        break

print("\nTraining Completed Successfully!")


In [ ]:
# ==========================================================
# PART 7 : LOAD BEST MODEL
# ==========================================================

model = ResUNet().to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

model.eval()

print("Best model loaded successfully!")


In [ ]:
# ==========================================================
# TEST FUNCTION
# ==========================================================

def test_model(model, loader, criterion, device):

    model.eval()

    test_loss = 0.0
    test_dice = 0.0
    test_iou = 0.0

    with torch.no_grad():

        progress_bar = tqdm(loader, desc="Testing")

        for images, masks, _ in progress_bar:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            dice, iou = calculate_metrics(outputs, masks)

            test_loss += loss.item()
            test_dice += dice
            test_iou += iou

    test_loss /= len(loader)
    test_dice /= len(loader)
    test_iou /= len(loader)

    return test_loss, test_dice, test_iou


In [ ]:
# ==========================================================
# EVALUATE MODEL
# ==========================================================

test_loss, test_dice, test_iou = test_model(
    model,
    test_loader,
    criterion,
    DEVICE
)

print("="*60)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Dice : {test_dice:.4f}")
print(f"Test IoU  : {test_iou:.4f}")

print("="*60)


In [ ]:
# ==========================================================
# SAVE RESULTS
# ==========================================================

results = {
    "Test Loss": test_loss,
    "Test Dice": test_dice,
    "Test IoU": test_iou
}

print(results)


In [ ]:
# PART 8 : VISUALIZATION

import matplotlib.pyplot as plt
import random

In [ ]:
# TRAINING CURVES

plt.figure(figsize=(18,5))

# Loss
plt.subplot(1,3,1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Dice
plt.subplot(1,3,2)
plt.plot(history["train_dice"], label="Train")
plt.plot(history["val_dice"], label="Validation")
plt.title("Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()

# IoU
plt.subplot(1,3,3)
plt.plot(history["train_iou"], label="Train")
plt.plot(history["val_iou"], label="Validation")
plt.title("IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# VISUALIZE TEST PREDICTIONS (annotated with shape category)

model.eval()

NUM_SAMPLES = 10

indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

plt.figure(figsize=(12, NUM_SAMPLES*4))

for i, idx in enumerate(indices):

    image, mask, label = test_dataset[idx]

    input_tensor = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        output = model(input_tensor)

        prediction = (torch.sigmoid(output) > 0.5).float()

    image_np = image.squeeze().cpu().numpy()
    mask_np = mask.squeeze().cpu().numpy()
    prediction_np = prediction.squeeze().cpu().numpy()

    shape_name = CLASS_NAMES[int(label)]

    # Original Image
    plt.subplot(NUM_SAMPLES, 3, 3*i+1)
    plt.imshow(image_np, cmap="gray")
    plt.title(f"Original (Shape: {shape_name})")
    plt.axis("off")

    # Ground Truth
    plt.subplot(NUM_SAMPLES, 3, 3*i+2)
    plt.imshow(mask_np, cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    # Prediction
    plt.subplot(NUM_SAMPLES, 3, 3*i+3)
    plt.imshow(prediction_np, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================
# OVERLAY PREDICTIONS (annotated with shape category)
# ==========================================================

NUM_SAMPLES = 5

indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

plt.figure(figsize=(15, NUM_SAMPLES*4))

for i, idx in enumerate(indices):

    image, mask, label = test_dataset[idx]

    image_np = image.squeeze().numpy()

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(DEVICE))
        prediction = (torch.sigmoid(output) > 0.5).float().squeeze().cpu().numpy()

    shape_name = CLASS_NAMES[int(label)]

    # Original
    plt.subplot(NUM_SAMPLES,2,2*i+1)
    plt.imshow(image_np, cmap="gray")
    plt.title(f"Original (Shape: {shape_name})")
    plt.axis("off")

    # Overlay
    plt.subplot(NUM_SAMPLES,2,2*i+2)
    plt.imshow(image_np, cmap="gray")
    plt.imshow(prediction, cmap="jet", alpha=0.4)
    plt.title("Prediction Overlay")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    total_dice = 0
    total_iou = 0

    all_preds = []
    all_masks = []

    with torch.no_grad():

        for images, masks, _ in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            predictions = (torch.sigmoid(outputs) > 0.5).float()

            dice = dice_coefficient(predictions, masks)
            iou = iou_score(predictions, masks)

            total_loss += loss.item()
            total_dice += dice.item()
            total_iou += iou.item()

            all_preds.extend(predictions.cpu().numpy().flatten())
            all_masks.extend(masks.cpu().numpy().flatten())

    precision = precision_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    results = {
        "Loss": total_loss / len(loader),
        "Dice": total_dice / len(loader),
        "IoU": total_iou / len(loader),
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

    return results


In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()


In [ ]:
train_results = evaluate_model(
    model,
    train_loader,
    criterion,
    DEVICE
)

val_results = evaluate_model(
    model,
    val_loader,
    criterion,
    DEVICE
)

test_results = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE
)


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Metric": ["Loss", "Dice", "IoU", "Precision", "Recall", "F1 Score"],
    "Train": [
        train_results["Loss"],
        train_results["Dice"],
        train_results["IoU"],
        train_results["Precision"],
        train_results["Recall"],
        train_results["F1 Score"]
    ],
    "Validation": [
        val_results["Loss"],
        val_results["Dice"],
        val_results["IoU"],
        val_results["Precision"],
        val_results["Recall"],
        val_results["F1 Score"]
    ],
    "Test": [
        test_results["Loss"],
        test_results["Dice"],
        test_results["IoU"],
        test_results["Precision"],
        test_results["Recall"],
        test_results["F1 Score"]
    ]
})

# Round to 4 decimal places
results_df = results_df.round(4)

# Display as a nice table
display(results_df)


In [ ]:
# ==========================================================
# SEGMENTATION QUALITY PER SHAPE CATEGORY
# ==========================================================
# Runs the model on a loader and computes Dice / IoU PER SAMPLE (rather
# than averaged over the whole batch), then groups those per-sample
# scores by the shape label (Irregular / Oval / Lobulated) so we can see
# which shape the model segments best/worst. The shape label is only
# used here to group results -- the model itself never sees it.

def dice_per_sample(pred, target, smooth=1e-6):
    pred = pred.contiguous().view(pred.shape[0], -1)
    target = target.contiguous().view(target.shape[0], -1)
    intersection = (pred * target).sum(dim=1)
    dice = (2.0 * intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)
    return dice

def iou_per_sample(pred, target, smooth=1e-6):
    pred = pred.contiguous().view(pred.shape[0], -1)
    target = target.contiguous().view(target.shape[0], -1)
    intersection = (pred * target).sum(dim=1)
    union = pred.sum(dim=1) + target.sum(dim=1) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou

def evaluate_by_shape(model, loader, device):

    model.eval()

    records = []

    with torch.no_grad():

        for images, masks, labels_batch in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            predictions = (torch.sigmoid(outputs) > 0.5).float()

            dice_vals = dice_per_sample(predictions, masks).cpu().numpy()
            iou_vals = iou_per_sample(predictions, masks).cpu().numpy()
            label_vals = labels_batch.numpy()

            for d, i, l in zip(dice_vals, iou_vals, label_vals):
                records.append({
                    "Shape": CLASS_NAMES[int(l)],
                    "Dice": float(d),
                    "IoU": float(i)
                })

    return pd.DataFrame.from_records(records)


In [ ]:
# Per-sample Dice/IoU on the test set, grouped by shape category

per_sample_df = evaluate_by_shape(model, test_loader, DEVICE)

shape_summary = per_sample_df.groupby("Shape")[["Dice", "IoU"]].agg(["mean", "std", "count"])
shape_summary = shape_summary.round(4)

print("Segmentation quality by shape category (Test Set)\n")
display(shape_summary)


In [ ]:
# Bar chart: mean Dice / IoU per shape category

grouped_mean = per_sample_df.groupby("Shape")[["Dice", "IoU"]].mean().reindex(
    [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
)

grouped_mean.plot(kind="bar", figsize=(8, 5), rot=0)
plt.title("Mean Segmentation Score by Shape Category (Test Set)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()


In [ ]:
#Save Model
torch.save(model.state_dict(), "ResUNet_CBIS_DDSM_Best.pth")